# SPIDER — Notebook 3: evaluation to journal standard

Everything in Tier 1 of the review-readiness list:

| section | what it answers |
|---|---|
| repeated 5x5 CV | are the results an artefact of one fold assignment? |
| leave-one-scanner-out | does the descriptor transfer to an unseen site? |
| fine-tuned CNN | was the learned arm represented fairly? |
| PR-AUC, MCC, calibration | do the results hold under 14-59% prevalence? |
| Benjamini-Hochberg | are the comparisons significant after multiplicity? |

Run Notebooks 1 and 2 first. **T4 GPU required** for the CNN section.

## Cell 1 — Load everything, aligned

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/spider'
CROP_TAG = 'tight'                      # which crop set these results describe

prim = pd.read_csv(f'{DRIVE}/spider_primary_cohort.csv')
z = np.load(f'{DRIVE}/spider_crops_{CROP_TAG}.npz', allow_pickle=True)
X_img, row_index = z['images'], z['row_index']
E = np.load(f'{DRIVE}/spider_dinov2_{CROP_TAG}.npy')

prim = prim.loc[row_index].reset_index(drop=True)
assert len(prim) == len(E) == len(X_img), "cohort / embeddings / crops misaligned"
print(f"aligned: {len(prim)} discs, {prim.Patient.nunique()} patients, "
      f"embeddings {E.shape}, crops {X_img.shape}")

# ---- scanner model, for leave-one-scanner-out ----
if not os.path.exists('overview.csv'):
    !wget -q -O overview.csv "https://zenodo.org/records/10159290/files/overview.csv?download=1"
ov = pd.read_csv('overview.csv')
ov['Patient'] = ov['new_file_name'].str.split('_').str[0].astype(int)
ov['sex'] = ov['sex'].astype(str).str.strip()          # 'F ' and 'F' were split
scan = (ov.groupby('Patient')
          .agg(scanner=('ManufacturerModelName', lambda s: s.mode().iat[0]),
               field=('MagneticFieldStrength', 'first'),
               sex=('sex', 'first')).reset_index())
prim = prim.merge(scan, on='Patient', how='left')

print("\nscanner distribution (patients):")
print(prim.drop_duplicates('Patient').scanner.value_counts().to_string())

INV = ['disc_height_ratio', 'ap_ratio', 'canal_ratio', 'volume_ratio', 'slip_pct']
TARGETS = {'disc narrowing': 'narrowing', 'disc bulging': 'bulging',
           'Pfirrmann >= 4': 'pfirrmann_severe', 'Modic (any)': 'modic_any'}
print("\nprevalence:", {k: f"{100*prim[v].mean():.0f}%" for k, v in TARGETS.items()})

## Cell 2 — Shared machinery

One place for the model definitions and the leakage-safe per-level z-scoring, so
every section below is running identical code on different splits.

Decision thresholds are chosen by Youden's J **on the training fold** and applied
to the held-out fold, so the MCC and calibration numbers carry no threshold
leakage — the caveat you flagged in the BUU protocol section.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             matthews_corrcoef, brier_score_loss, roc_curve)

SEED, N_PCA = 0, 128
levels_all = prim['ivd_label'].values


def _zscore(ref, ref_lvl, X, lvl):
    out = X.copy()
    gm, gs = ref.mean(), ref.std().replace(0, 1)
    for l in np.unique(lvl):
        m = ref_lvl == l
        mu = ref[m].mean() if m.sum() >= 20 else gm
        sd = ref[m].std().replace(0, 1) if m.sum() >= 20 else gs
        out.loc[lvl == l] = (X.loc[lvl == l] - mu) / sd
    return out


def _lr(seed=SEED):
    return LogisticRegression(max_iter=5000, class_weight='balanced',
                              random_state=seed)


def fit_predict(arm, tr, te, y, seed=SEED):
    # Train one arm on tr; return (test probabilities, train probabilities)
    lvl_tr, lvl_te = levels_all[tr], levels_all[te]
    if arm == 'geometry':
        A, B = prim[INV].iloc[tr], prim[INV].iloc[te]
        A2 = _zscore(A, lvl_tr, A, lvl_tr)
        B2 = _zscore(A, lvl_tr, B, lvl_te)
        m = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                          _lr(seed)).fit(A2, y[tr])
        return m.predict_proba(B2)[:, 1], m.predict_proba(A2)[:, 1]
    if arm == 'dinov2':
        m = make_pipeline(StandardScaler(),
                          PCA(n_components=min(N_PCA, len(tr) - 1),
                              random_state=seed), _lr(seed)).fit(E[tr], y[tr])
        return m.predict_proba(E[te])[:, 1], m.predict_proba(E[tr])[:, 1]
    if arm == 'fusion':
        A, B = prim[INV].iloc[tr], prim[INV].iloc[te]
        A2 = _zscore(A, lvl_tr, A, lvl_tr).values
        B2 = _zscore(A, lvl_tr, B, lvl_te).values
        pca = make_pipeline(StandardScaler(),
                            PCA(n_components=min(32, len(tr) - 1),
                                random_state=seed)).fit(E[tr])
        Atr = np.hstack([A2, pca.transform(E[tr])])
        Ate = np.hstack([B2, pca.transform(E[te])])
        w = np.where(y[tr] == 1,
                     (y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1), 1.)
        g = HistGradientBoostingClassifier(max_depth=3, max_iter=300,
                                           random_state=seed)
        g.fit(Atr, y[tr], sample_weight=w)
        return g.predict_proba(Ate)[:, 1], g.predict_proba(Atr)[:, 1]
    raise ValueError(arm)


def run_split(arm, y, splits, seed=SEED):
    # Out-of-fold probabilities + binary calls at train-selected thresholds
    p = np.zeros(len(y)); b = np.zeros(len(y), int)
    for tr, te in splits:
        pte, ptr = fit_predict(arm, tr, te, y, seed)
        fpr, tpr, thr = roc_curve(y[tr], ptr)
        t = thr[np.argmax(tpr - fpr)]            # Youden J on TRAIN only
        p[te], b[te] = pte, (pte >= t).astype(int)
    return p, b


def metrics(y, p, b):
    return dict(auc=roc_auc_score(y, p), pr_auc=average_precision_score(y, p),
                mcc=matthews_corrcoef(y, b), brier=brier_score_loss(y, p))


ARMS = ['geometry', 'dinov2', 'fusion']
print("machinery ready")

## Cell 3 — Repeated 5x5 cross-validation

Five repeats of stratified, patient-grouped 5-fold — 25 fits per arm per target.
Reporting the spread across repeats shows whether any difference survives
fold-assignment luck, the same standard as the BUU extended-descriptor study.

In [ ]:
N_REPEATS = 5
OOF_FILE = f'{DRIVE}/nb3_oof_rep.npy'
oof_rep = (np.load(OOF_FILE, allow_pickle=True).item()
           if os.path.exists(OOF_FILE) else {})
if oof_rep:
    print(f"loaded {len(oof_rep)} cached out-of-fold runs from Drive")
rows = []
t0 = time.time()

for tname, tcol in TARGETS.items():
    y = prim[tcol].astype(int).values
    for arm in ARMS:
        per_repeat = []
        for rep in range(N_REPEATS):
            sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True,
                                        random_state=rep)
            splits = list(sgkf.split(np.zeros(len(y)), y, prim.Patient.values))
            key = (tname, arm, rep)
            if key not in oof_rep:
                oof_rep[key] = run_split(arm, y, splits, seed=rep)
            p, b = oof_rep[key]
            per_repeat.append(metrics(y, p, b))
        df = pd.DataFrame(per_repeat)
        rows.append(dict(target=tname, arm=arm,
                         **{f'{k}_mean': df[k].mean() for k in df},
                         **{f'{k}_sd': df[k].std() for k in df}))
        print(f"  {tname:16s} {arm:10s} "
              f"AUC {df.auc.mean():.3f}+/-{df.auc.std():.3f}  "
              f"PR {df.pr_auc.mean():.3f}  MCC {df.mcc.mean():.3f}  "
              f"({time.time()-t0:.0f}s)", flush=True)

rep_res = pd.DataFrame(rows)
print("\nRepeated 5x5 CV -- AUC (mean +/- SD over 5 repeats)")
print("=" * 64)
print(rep_res.pivot(index='target', columns='arm', values='auc_mean')
      .round(3).to_string())
print("\nrepeat-to-repeat SD:")
print(rep_res.pivot(index='target', columns='arm', values='auc_sd')
      .round(4).to_string())
rep_res.to_csv(f'{DRIVE}/nb3_repeated_cv.csv', index=False)
np.save(OOF_FILE, oof_rep, allow_pickle=True)
oof_main = {(t, a): oof_rep[(t, a, 0)] for (t, a, r) in oof_rep if r == 0}
print(f"\n{len(oof_rep)} out-of-fold runs saved to Drive")

## Cell 4 — Prevalence-appropriate metrics and calibration

AUC is prevalence-insensitive, which flatters a model on a 14%-positive target.
PR-AUC and MCC are not. Brier score and the calibration curve say whether the
probabilities mean anything, which matters if the output is ever to inform a
clinical decision rather than rank a list.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

print("Full metric table (repeat means)")
print("=" * 78)
for tname, tcol in TARGETS.items():
    prev = 100 * prim[tcol].mean()
    sub = rep_res[rep_res.target == tname].set_index('arm')
    print(f"\n{tname}  (prevalence {prev:.0f}%, "
          f"PR-AUC baseline {prev/100:.2f})")
    print(sub[['auc_mean', 'pr_auc_mean', 'mcc_mean', 'brier_mean']]
          .round(3).to_string())

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, (tname, tcol) in zip(axes, TARGETS.items()):
    y = prim[tcol].astype(int).values
    for arm in ARMS:
        p, _ = oof_main[(tname, arm)]
        fr, mp = calibration_curve(y, p, n_bins=8, strategy='quantile')
        ax.plot(mp, fr, marker='o', ms=3, label=arm)
    ax.plot([0, 1], [0, 1], 'k--', lw=.8)
    ax.set_title(tname, fontsize=9)
    ax.set_xlabel('predicted'); ax.set_ylabel('observed')
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{DRIVE}/nb3_calibration.png', dpi=150)
plt.show()

## Cell 5 — Leave-one-scanner-out

The strongest generalisation evidence available without a second cohort. Each
scanner model is held out entirely: a model is trained on the others and tested
on it. If the descriptor transfers, its held-out AUC stays near the pooled
figure; if it were fitting site-specific artefacts, it would collapse.

In [ ]:
MIN_PATIENTS = 15
counts = prim.drop_duplicates('Patient').scanner.value_counts()
held_out = [s for s, n in counts.items() if n >= MIN_PATIENTS]
print(f"scanners held out one at a time: {held_out}")
print(f"(others pooled into training only: "
      f"{[s for s in counts.index if s not in held_out]})\n")

rows = []
for tname, tcol in TARGETS.items():
    y = prim[tcol].astype(int).values
    for s in held_out:
        te = np.flatnonzero((prim.scanner == s).values)
        tr = np.flatnonzero((prim.scanner != s).values)
        if len(np.unique(y[te])) < 2:
            continue
        for arm in ARMS:
            pte, ptr = fit_predict(arm, tr, te, y)
            fpr, tpr, thr = roc_curve(y[tr], ptr)
            b = (pte >= thr[np.argmax(tpr - fpr)]).astype(int)
            m = metrics(y[te], pte, b)
            rows.append(dict(target=tname, scanner=s, arm=arm, n_test=len(te),
                             n_patients=len(np.unique(prim.Patient.values[te])),
                             pos=int(y[te].sum()), **m))

los = pd.DataFrame(rows)
print("Leave-one-scanner-out AUC")
print("=" * 70)
for tname in TARGETS:
    print(f"\n{tname}")
    print(los[los.target == tname]
          .pivot(index='scanner', columns='arm', values='auc')
          .round(3).to_string())

print("\ntest-set sizes (patients / discs):")
print(los.drop_duplicates('scanner').set_index('scanner')[['n_patients', 'n_test']]
      .to_string())
print("\nmean across held-out scanners:")
print(los.groupby(['target', 'arm']).auc.mean().unstack().round(3).to_string())
los.to_csv(f'{DRIVE}/nb3_leave_one_scanner_out.csv', index=False)

## Cell 6 — Fine-tuned CNN, all targets, five seeds (checkpointed)

Same five repeats and the **same fold assignment per repeat** as Cell 3, so every
CNN run is paired with a geometry run on identical splits. A new seed per repeat
and per fold means the spread reported includes the CNN's own training
non-determinism, which is what flipped the verdict between your two earlier runs.

Every (target, repeat) is saved to Drive the moment it finishes. If Colab drops,
re-run Cells 1-2 then this cell: finished runs are skipped. Roughly 60-70 minutes
on a T4 for everything; you can stop and resume at any point.

In [ ]:
import torch, torch.nn as nn, torchvision
from torch.utils.data import TensorDataset, DataLoader

CNN_EPOCHS = 8
N_REPEATS = 5
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', "switch to a T4 GPU: Runtime > Change runtime type"
Xt = torch.from_numpy(X_img).float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
CKPT = f'{DRIVE}/nb3_cnn'
os.makedirs(CKPT, exist_ok=True)


def train_cnn(tr, te, y, seed):
    # returns (test probabilities, train probabilities)
    torch.manual_seed(seed); np.random.seed(seed)
    net = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1')
    net.classifier[1] = nn.Linear(net.classifier[1].in_features, 1)
    net = net.to(dev)
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(TensorDataset(Xt[tr], torch.tensor(y[tr]).float()),
                    batch_size=32, shuffle=True, generator=g)
    pw = torch.tensor([(y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)],
                      dtype=torch.float32).to(dev)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CNN_EPOCHS)
    net.train()
    for _ in range(CNN_EPOCHS):
        for xb, yb in dl:
            opt.zero_grad()
            lossf(net(xb.to(dev)).squeeze(1), yb.to(dev)).backward()
            opt.step()
        sched.step()
    net.eval()
    def infer(idx):
        out = []
        with torch.no_grad():
            for i in range(0, len(idx), 64):
                out.append(torch.sigmoid(
                    net(Xt[idx[i:i+64]].to(dev)).squeeze(1)).cpu().numpy())
        return np.concatenate(out)
    pte, ptr = infer(te), infer(tr)
    del net; torch.cuda.empty_cache()
    return pte, ptr


cnn_rep = {}
t_all = time.time()
for tname, tcol in TARGETS.items():
    y = prim[tcol].astype(int).values
    for rep in range(N_REPEATS):
        f = f'{CKPT}/{tcol}_rep{rep}.npz'
        if os.path.exists(f):
            d = np.load(f)
            cnn_rep[(tname, 'cnn', rep)] = (d['p'], d['b'])
            print(f"  {tname:16s} rep {rep}  cached", flush=True)
            continue
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=rep)
        splits = list(sgkf.split(np.zeros(len(y)), y, prim.Patient.values))
        p = np.zeros(len(y)); b = np.zeros(len(y), int)
        t0 = time.time()
        for k, (tr, te) in enumerate(splits):
            pte, ptr = train_cnn(tr, te, y, seed=1000 * rep + k)
            fpr, tpr, thr = roc_curve(y[tr], ptr)
            p[te] = pte
            b[te] = (pte >= thr[np.argmax(tpr - fpr)]).astype(int)
        np.savez(f, p=p, b=b)
        cnn_rep[(tname, 'cnn', rep)] = (p, b)
        print(f"  {tname:16s} rep {rep}  AUC {roc_auc_score(y, p):.3f}  "
              f"({time.time()-t0:.0f}s, total {(time.time()-t_all)/60:.1f} min)",
              flush=True)

print(f"\ncomplete: {len(cnn_rep)} of {len(TARGETS)*N_REPEATS} CNN runs")

## Cell 7 — CNN on held-out scanners (checkpointed)

The test that decides the transfer argument. Geometry lost essentially nothing on
unseen scanners while frozen DINOv2 lost up to 0.075. If the fine-tuned CNN also
degrades, measurement has a transfer advantage; if not, that argument weakens.
One seed per split — about 15 minutes.

In [ ]:
counts = prim.drop_duplicates('Patient').scanner.value_counts()
held_out = [s for s, n in counts.items() if n >= 15]
cnn_los = []
t0 = time.time()
for tname, tcol in TARGETS.items():
    y = prim[tcol].astype(int).values
    for s in held_out:
        te = np.flatnonzero((prim.scanner == s).values)
        tr = np.flatnonzero((prim.scanner != s).values)
        if len(np.unique(y[te])) < 2:
            continue
        f = f'{CKPT}/loso_{tcol}_{s}.npz'
        if os.path.exists(f):
            d = np.load(f); pte, b = d['p'], d['b']
        else:
            pte, ptr = train_cnn(tr, te, y, seed=7)
            fpr, tpr, thr = roc_curve(y[tr], ptr)
            b = (pte >= thr[np.argmax(tpr - fpr)]).astype(int)
            np.savez(f, p=pte, b=b)
        m = metrics(y[te], pte, b)
        cnn_los.append(dict(target=tname, scanner=s, arm='cnn', n_test=len(te),
                            n_patients=len(np.unique(prim.Patient.values[te])),
                            pos=int(y[te].sum()), **m))
        print(f"  {tname:16s} {s:12s} AUC {m['auc']:.3f}  "
              f"({(time.time()-t0)/60:.1f} min)", flush=True)
cnn_los = pd.DataFrame(cnn_los)

## Cell 8 — Combined tables, all four arms

In [ ]:
ALL_ARMS = ['geometry', 'dinov2', 'fusion', 'cnn']

# ---- repeated CV, including the CNN ----
rows = []
for tname, tcol in TARGETS.items():
    y = prim[tcol].astype(int).values
    for arm in ALL_ARMS:
        src = cnn_rep if arm == 'cnn' else oof_rep
        reps = [metrics(y, *src[(tname, arm, r)]) for r in range(N_REPEATS)
                if (tname, arm, r) in src]
        if not reps:
            continue
        df = pd.DataFrame(reps)
        rows.append(dict(target=tname, arm=arm, n_runs=len(reps),
                         **{f'{k}': df[k].mean() for k in df},
                         **{f'{k}_sd': df[k].std() for k in df}))
full = pd.DataFrame(rows)

print("Repeated 5x5 CV, all arms -- AUC mean (SD across repeats)")
print("=" * 72)
tab = full.pivot(index='target', columns='arm', values='auc')[ALL_ARMS]
sd = full.pivot(index='target', columns='arm', values='auc_sd')[ALL_ARMS]
print(tab.round(3).astype(str).add(' (').add(sd.round(3).astype(str)).add(')')
      .to_string())

for met in ['pr_auc', 'mcc', 'brier']:
    print(f"\n{met}:")
    print(full.pivot(index='target', columns='arm', values=met)[ALL_ARMS]
          .round(3).to_string())
full.to_csv(f'{DRIVE}/nb3_all_arms_repeated_cv.csv', index=False)

# ---- scanner transfer, including the CNN ----
if len(cnn_los):
    los_all = pd.concat([los, cnn_los], ignore_index=True)
    print("\n" + "=" * 72)
    print("Leave-one-scanner-out, mean AUC across held-out scanners")
    loso_mean = los_all.groupby(['target', 'arm']).auc.mean().unstack()[ALL_ARMS]
    print(loso_mean.round(3).to_string())

    print("\nTRANSFER GAP = held-out-scanner AUC minus pooled CV AUC")
    print("(near zero = transfers; negative = degrades on an unseen site)")
    print((loso_mean - tab).round(3).to_string())
    los_all.to_csv(f'{DRIVE}/nb3_loso_all_arms.csv', index=False)

## Cell 9 — Significance across repeats, with multiplicity control

For each comparison, a patient-level bootstrap is drawn **within each of the five
repeats** on its paired geometry run, and the five distributions are pooled. The
resulting interval therefore carries both sampling variance and run-to-run
variance — so a verdict cannot hinge on one lucky or unlucky training run.
Benjamini-Hochberg is then applied across all comparisons.

In [ ]:
N_BOOT_PER_REP = 400
rng = np.random.default_rng(SEED)
pats, uniq = prim.Patient.values, prim.Patient.unique()
boot = [np.concatenate([np.flatnonzero(pats == u)
                        for u in rng.choice(uniq, len(uniq), replace=True)])
        for _ in range(N_BOOT_PER_REP)]

rows = []
t0 = time.time()
for tname, tcol in TARGETS.items():
    y = prim[tcol].astype(int).values
    for arm in ['dinov2', 'fusion', 'cnn']:
        src = cnn_rep if arm == 'cnn' else oof_rep
        dist, per_rep = [], []
        for r in range(N_REPEATS):
            if (tname, arm, r) not in src:
                continue
            base, p = oof_rep[(tname, 'geometry', r)][0], src[(tname, arm, r)][0]
            per_rep.append(roc_auc_score(y, p) - roc_auc_score(y, base))
            dist += [roc_auc_score(y[i], p[i]) - roc_auc_score(y[i], base[i])
                     for i in boot if len(np.unique(y[i])) > 1]
        if not dist:
            continue
        dist = np.array(dist)
        lo, hi = np.percentile(dist, [2.5, 97.5])
        pval = 2 * min((dist <= 0).mean(), (dist >= 0).mean())
        rows.append(dict(target=tname, arm=arm, n_reps=len(per_rep),
                         dauc=np.mean(per_rep), dauc_rep_sd=np.std(per_rep),
                         lo=lo, hi=hi, p=max(pval, 1 / len(dist))))
    print(f"  {tname} done ({time.time()-t0:.0f}s)", flush=True)

sig = pd.DataFrame(rows).sort_values('p').reset_index(drop=True)
m = len(sig)
adj = sig.p * m / (sig.index + 1)
sig['p_bh'] = adj[::-1].cummin()[::-1].clip(upper=1)     # step-up, largest rank first
sig['verdict'] = np.where(sig.p_bh >= .05, 'indistinguishable',
                          np.where(sig.dauc < 0, 'geometry better',
                                   'this arm better'))

print("\ndAUC vs geometry: mean over repeats, pooled-bootstrap 95% CI, BH-adjusted")
print("=" * 88)
print(sig[['target', 'arm', 'n_reps', 'dauc', 'dauc_rep_sd', 'lo', 'hi', 'p', 'p_bh',
           'verdict']].round(4).to_string(index=False))
sig.to_csv(f'{DRIVE}/nb3_significance_all_arms.csv', index=False)

print("\n--- written to Drive ---")
for f in ['nb3_oof_rep.npy', 'nb3_cnn/ (per-run checkpoints)',
          'nb3_all_arms_repeated_cv.csv', 'nb3_loso_all_arms.csv',
          'nb3_significance_all_arms.csv', 'nb3_calibration.png']:
    print("  " + f)